# Reproduce paper figures

This notebook regenerates paper figures from the released HEC package:


Data sources:
- `../Frontend/data/hec-dataset-v1.0.csv` (composition database)
- `data/distance.csv` (precomputed most-dissimilar element distances)
- `data/element-vectors.csv` (element embedding vectors used for t-SNE)
- `data/rdf-wasserstein-normalized.csv` (RDF Wasserstein metrics for relaxed supercells)

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from pymatviz.ptable import ptable_heatmap

ROOT = Path('.').resolve()
DATA_CSV = ROOT.parent / 'Frontend' / 'data' / 'hec-dataset-v1.0.csv'
ANALYSIS_DATA = ROOT / 'data'
DISTANCE_CSV = ANALYSIS_DATA / 'distance.csv'
ELEMENT_VECTORS_CSV = ANALYSIS_DATA / 'element-vectors.csv'
RDF_WASSERSTEIN_CSV = ANALYSIS_DATA / 'rdf-wasserstein-normalized.csv'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 24,
    'axes.titleweight': 'bold',
})


def prepare_pie_counts(counts, top_n=7, extra_others=0):
    """Group counts into top-N categories plus an Others bucket."""
    top = counts.head(top_n)
    others_sum = counts.iloc[top_n:].sum() + extra_others
    if others_sum > 0:
        return pd.concat([top, pd.Series([others_sum], index=['Others'])])
    return counts


def _pie_font_sizes(compact=False):
    return {
        'pct': 9 if compact else 11,
        'center': 14 if compact else 16,
        'legend': 9 if compact else 11,
        'legend_title': 10 if compact else 13,
        'panel': 13 if compact else 14,
    }


def draw_donut_pie(ax, counts, top_n=7, extra_others=0, compact=False):
    """Draw a donut pie and return legend handles/labels."""
    final_counts = prepare_pie_counts(counts, top_n=top_n, extra_others=extra_others)

    labels = final_counts.index.tolist()
    sizes = final_counts.values
    total = sizes.sum()
    fonts = _pie_font_sizes(compact)

    cmap = plt.get_cmap('tab20')
    colors = [cmap(i % 20) for i in range(len(sizes))]

    wedges, _, autotexts = ax.pie(
        sizes,
        colors=colors,
        startangle=90,
        counterclock=False,
        autopct=lambda pct: f'{pct:.1f}%' if pct >= 4 else '',
        pctdistance=0.82,
        wedgeprops=dict(width=0.42, edgecolor='white', linewidth=2.5, antialiased=True),
    )

    for t in autotexts:
        t.set_color('white')
        t.set_fontsize(fonts['pct'])
        t.set_fontweight('bold')

    ax.text(
        0, 0, f'{total:,}\nentries',
        ha='center', va='center',
        fontsize=fonts['center'], fontweight='bold', color='#333333',
    )

    legend_labels = [
        f'{label}  ({value:,}, {100 * value / total:.1f}%)'
        for label, value in zip(labels, sizes)
    ]

    ax.set_aspect('equal')
    ax.axis('off')
    return final_counts, wedges, legend_labels, fonts


def plot_combined_pie_panels(panels, filename, figsize=(6.2, 5.6)):
    """Render multiple donut pies in a tight vertical stack with equal panel sizes."""
    n = len(panels)
    fig = plt.figure(figsize=figsize, facecolor='white')
    gs = fig.add_gridspec(
        n,
        2,
        width_ratios=[0.78, 1.22],
        height_ratios=[1] * n,
        hspace=0.04,
        wspace=0.0,
        left=0.10,
        right=0.99,
        top=0.99,
        bottom=0.01,
    )

    results = []
    for i, panel in enumerate(panels):
        ax_pie = fig.add_subplot(gs[i, 0])
        ax_leg = fig.add_subplot(gs[i, 1])
        ax_pie.set_box_aspect(1)

        final_counts, wedges, legend_labels, fonts = draw_donut_pie(
            ax_pie,
            panel['counts'],
            top_n=panel.get('top_n', 7),
            extra_others=panel.get('extra_others', 0),
            compact=True,
        )

        panel_label = panel.get('panel_label')
        if panel_label:
            ax_pie.text(
                -0.05, 1.02, panel_label,
                transform=ax_pie.transAxes,
                fontsize=fonts['panel'], fontweight='bold', va='bottom', ha='left',
            )

        ax_leg.axis('off')
        ax_leg.legend(
            wedges,
            legend_labels,
            title='Category',
            loc='center left',
            borderaxespad=0.0,
            frameon=True,
            fancybox=True,
            shadow=True,
            fontsize=fonts['legend'],
            title_fontsize=fonts['legend_title'],
        )
        results.append(final_counts)

    fig.savefig(filename, dpi=200, facecolor='white', pad_inches=0.02)
    plt.show()
    return results

print('DATA_CSV =', DATA_CSV)
print('DISTANCE_CSV =', DISTANCE_CSV)
print('ELEMENT_VECTORS_CSV =', ELEMENT_VECTORS_CSV)
print('RDF_WASSERSTEIN_CSV =', RDF_WASSERSTEIN_CSV)
print('OUTPUT_DIR =', OUTPUT_DIR)

## Load released dataset

In [ ]:
df = pd.read_csv(DATA_CSV, encoding='utf-8-sig')
assert len(df) == 717, f'Expected 717 compositions, got {len(df)}'
assert 'Sorted Json formula' in df.columns
assert 'prototype_name' in df.columns
df.head()

## Derive anion and cation sites

In [ ]:
df = df.copy()
df['Anion'] = None
df['Cation'] = None

for idx in df.index:
    formula = json.loads(df.at[idx, 'Sorted Json formula'])

    last_key = list(formula.keys())[-1]
    anions = formula[last_key]
    sorted_anions = dict(sorted(anions.items(), key=lambda item: item[1], reverse=True))
    df.at[idx, 'Anion'] = list(sorted_anions.keys())[0]

    for site_value in formula.values():
        if (
            len(site_value) >= 3
            and 'C' not in site_value.keys()
            and 'Te' not in site_value.keys()
        ):
            df.at[idx, 'Cation'] = json.dumps(site_value)

print('Anion top counts:')
print(df['Anion'].value_counts().head(10))
print('\nprototype_name top counts:')
print(df['prototype_name'].value_counts(dropna=False).head(12))

## Figure 3 — anion and crystal-structure distributions

In [ ]:
pie_path = OUTPUT_DIR / 'pie_chart_anions_structure.png'
unlabeled_count = int(df['prototype_name'].isna().sum())

anion_counts, structure_counts = plot_combined_pie_panels(
    panels=[
        {
            'counts': df['Anion'].value_counts(),
            'top_n': 7,
            'panel_label': '(a)',
        },
        {
            'counts': df['prototype_name'].value_counts(),
            'top_n': 9,
            'extra_others': unlabeled_count,
            'panel_label': '(b)',
        },
    ],
    filename=str(pie_path),
)

print('Saved', pie_path)
print('Panel (a):')
print(anion_counts)
print('\nPanel (b):')
print(structure_counts)

## Figure 4 — element frequency in high-entropy oxides

In [ ]:
O_SITE_ELEMENTS = {'O', 'F', 'Cl', 'Br', 'I'}

df_oxides = df[df['Anion'] == 'O'].dropna(subset=['Cation'])
dict_o2 = {}
for idx in df_oxides.index:
    for element in json.loads(df_oxides.at[idx, 'Cation']):
        if element in O_SITE_ELEMENTS:
            continue
        dict_o2[element] = dict_o2.get(element, 0) + 1

top_oxides = sorted(dict_o2.items(), key=lambda item: item[1], reverse=True)[:10]
print('Oxide compositions with cation site:', len(df_oxides))
print('Top oxide cation frequencies:')
for element, count in top_oxides:
    print(f'  {element}: {count}')

In [ ]:
oxide_path = OUTPUT_DIR / 'O.png'

fig = ptable_heatmap(dict_o2, fmt='.0f')
fig.layout.title = dict(
    text='Element Frequency in High-entropy Oxides',
    x=0.45,
    y=0.96,
)
fig.show()
fig.write_image(str(oxide_path))
print('Saved', oxide_path)

## Figure 5a — t-SNE outlier map (composition index 142)

In [ ]:
from sklearn.manifold import TSNE

OUTLIER_INDEX = 142


def build_tsne_df(df_vector):
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    tsne_results = tsne.fit_transform(df_vector.values)
    return pd.DataFrame(tsne_results, columns=['TSNE-1', 'TSNE-2'], index=df_vector.index)


def plot_tsne_outlier(tsne_df, a_site_elements, outlier_element):
    other_a_site = [el for el in a_site_elements if el != outlier_element]
    background = [el for el in tsne_df.index if el not in a_site_elements]

    fig, ax = plt.subplots(figsize=(6, 6))

    ax.scatter(
        tsne_df.loc[background, 'TSNE-1'],
        tsne_df.loc[background, 'TSNE-2'],
        c='0.55',
        s=40,
        alpha=0.55,
        linewidths=0,
        label='Other elements',
        zorder=1,
    )
    ax.scatter(
        tsne_df.loc[other_a_site, 'TSNE-1'],
        tsne_df.loc[other_a_site, 'TSNE-2'],
        c='#1f77b4',
        s=90,
        alpha=0.95,
        edgecolors='white',
        linewidths=0.6,
        label='A-site elements',
        zorder=3,
    )
    ax.scatter(
        tsne_df.loc[[outlier_element], 'TSNE-1'],
        tsne_df.loc[[outlier_element], 'TSNE-2'],
        c='#d62728',
        s=140,
        alpha=1.0,
        edgecolors='black',
        linewidths=0.8,
        label=f'Outlier: {outlier_element}',
        zorder=4,
    )

    for element in tsne_df.index:
        x_coord = tsne_df.at[element, 'TSNE-1']
        y_coord = tsne_df.at[element, 'TSNE-2']
        if element == outlier_element:
            color, fontweight, fontsize = '#d62728', 'bold', 13.5
        elif element in other_a_site:
            color, fontweight, fontsize = '#1f77b4', 'normal', 12
        else:
            color, fontweight, fontsize = '0.45', 'normal', 10.5

        ax.annotate(
            text=element,
            xy=(x_coord, y_coord),
            xytext=(1, 1),
            textcoords='offset points',
            ha='left',
            va='bottom',
            fontsize=fontsize,
            fontweight=fontweight,
            color=color,
            alpha=0.9,
            zorder=5,
        )

    ax.set_xlabel('t-SNE Dimension 1')
    ax.set_ylabel('t-SNE Dimension 2')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(loc='best', frameon=True)
    fig.tight_layout()
    return fig


outlier_row = distance_df.loc[OUTLIER_INDEX]
outlier_element = outlier_row['Element']
distance = float(outlier_row['Distance'])
formula = json.loads(outlier_row['Sorted Json formula'])
a_site_elements = sorted(formula['A'].keys())

# Confirm this composition is still present in the released HEC dataset.
matches = df.index[df['Sorted Json formula'] == outlier_row['Sorted Json formula']].tolist()
assert matches, 'Outlier composition not found in released hec-dataset-v1.0.csv'

df_vector = pd.read_csv(ELEMENT_VECTORS_CSV, index_col=0)
tsne_df = build_tsne_df(df_vector)

tsne_path = OUTPUT_DIR / 'figure_tsne_outlier_142.png'
fig_tsne = plot_tsne_outlier(tsne_df, a_site_elements, outlier_element)
fig_tsne.savefig(tsne_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'Index: {OUTLIER_INDEX}')
print(f'A-site elements: {a_site_elements}')
print(f'Outlier element: {outlier_element} (distance={distance:.6f})')
print(f'Matched released dataset row index: {matches[0]}')
print('Saved', tsne_path)

## Figure 5b — embedding-centroid distance histogram


In [ ]:
def style_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', color='0.9', linewidth=0.8)
    ax.set_axisbelow(True)


def build_histogram_counts(distance_df, include_high_bin=True):
    if include_high_bin:
        bins = [0, 0.15, 0.20, 0.25, 0.30, 0.35, 1.0]
        labels = ['<0.15', '0.15-0.20', '0.20-0.25', '0.25-0.30', '0.30-0.35', '>0.35']
    else:
        bins = [0, 0.15, 0.20, 0.25, 0.30, 0.35]
        labels = ['<0.15', '0.15-0.20', '0.20-0.25', '0.25-0.30', '0.30-0.35']

    return (
        pd.cut(distance_df['Distance'], bins=bins, labels=labels, include_lowest=True)
        .value_counts()
        .sort_index()
    )


distance_df = pd.read_csv(DISTANCE_CSV, index_col=0)
hist_counts_plot = build_histogram_counts(distance_df, include_high_bin=False)

hist_path = OUTPUT_DIR / 'figure_distance_histogram.png'
fig_hist, ax_hist = plt.subplots(figsize=(3.7, 3.0), constrained_layout=True)
ax_hist.bar(
    hist_counts_plot.index.astype(str),
    hist_counts_plot.values,
    color='0.25',
    edgecolor='0.15',
)
ax_hist.set_xlabel('Element vector distance')
ax_hist.set_ylabel('Number of compositions')
ax_hist.tick_params(axis='x', rotation=35)
style_axes(ax_hist)
fig_hist.savefig(hist_path, dpi=600, bbox_inches='tight')
plt.show()

print('Saved', hist_path)
print('Histogram counts:')
print(hist_counts_plot.to_string())
print(f'Compositions in distance table: {len(distance_df)}')
print('Top flagged elements:')
print(distance_df['Element'].value_counts().head(10).to_string())

## Figure 6 — normalized RDF distortion by structure prototype

In [ ]:
import re

import numpy as np
from matplotlib.colors import LinearSegmentedColormap

ROCKSALT_CHALCOGENS = {'S', 'Se', 'Te'}
ROCKSALT_CARBIDE_NITRIDE = {'C', 'N'}
ROCKSALT_OXIDE = {'O'}
ROCKSALT_SUBCLASS_LABELS = {
    'oxide': 'rock-salt (oxide)',
    'carbide/nitride': 'rock-salt (carbide/nitride)',
    'chalcogenide': 'rock-salt (chalcogenide)',
}


def row_num_from_structure_key(structure_key):
    match = re.match(r'row(\d+)_', structure_key)
    return int(match.group(1)) if match else None


def rocksalt_anion_elements(formula_json):
    payload = json.loads(formula_json)
    elements = set()
    for site in ('B', 'C'):
        elements.update(payload.get(site, {}).keys())
    return elements


def rocksalt_chem_class(formula_json):
    elements = rocksalt_anion_elements(formula_json)
    if elements & ROCKSALT_CHALCOGENS:
        return 'chalcogenide'
    if elements & ROCKSALT_OXIDE:
        return 'oxide'
    if elements & ROCKSALT_CARBIDE_NITRIDE:
        return 'carbide/nitride'
    raise ValueError(f'Unable to classify rock-salt anion chemistry: {formula_json}')


def build_rocksalt_class_lookup(dataset_csv):
    dataset = pd.read_csv(dataset_csv, encoding='utf-8-sig')
    lookup = {}
    for row_idx, row in dataset.iterrows():
        if row['prototype_name'] != 'rock-salt':
            continue
        lookup[row_idx] = rocksalt_chem_class(row['Sorted Json formula'])
    return lookup


def split_rocksalt_prototypes(rdf_df, dataset_csv):
    result = rdf_df.copy()
    lookup = build_rocksalt_class_lookup(dataset_csv)
    rock_salt_mask = result['prototype'] == 'rock-salt'
    row_nums = result.loc[rock_salt_mask, 'structure_key'].map(row_num_from_structure_key)
    chem_classes = row_nums.map(lookup)
    if chem_classes.isna().any():
        missing = result.loc[rock_salt_mask].loc[chem_classes.isna(), 'structure_key'].tolist()
        raise ValueError(f'Missing rock-salt chemistry mapping for: {missing[:5]}')
    result.loc[rock_salt_mask, 'prototype'] = chem_classes.map(ROCKSALT_SUBCLASS_LABELS)
    return result


def format_prototype_label(prototype):
    label = prototype.strip()
    if re.search(r'--+', label):
        base, suffix = re.split(r'--+', label, maxsplit=1)
        label = f'{base.strip()} ({suffix.strip()})'
    if label:
        label = label[0].upper() + label[1:]
    if re.search(r'\d', label):
        math_label = re.sub(r'(\d+)', r'_\1', label)
        return rf'$\mathrm{{{math_label}}}$'
    return label


def prototype_summary(rdf_df, min_count):
    summary = (
        rdf_df.groupby('prototype')
        .agg(
            n=('norm_relative_first_peak', 'count'),
            median=('norm_relative_first_peak', 'median'),
            mean=('norm_relative_first_peak', 'mean'),
            q25=('norm_relative_first_peak', lambda x: x.quantile(0.25)),
            q75=('norm_relative_first_peak', lambda x: x.quantile(0.75)),
        )
        .reset_index()
    )
    return summary[summary['n'] >= min_count].sort_values('median', ascending=True)


def plot_panel_c_figure(rdf_df, output_dir, min_prototype_count=10, dataset_csv=None):
    if dataset_csv is not None:
        rdf_df = split_rocksalt_prototypes(rdf_df, dataset_csv)

    metric = 'norm_relative_first_peak'
    metric_label = 'Normalized RDF distortion index'
    metric_unit = r'($\it{r}$$_{\mathrm{1st peak}}$)$^{\mathrm{-1}}$'

    summary = prototype_summary(rdf_df, min_prototype_count)
    major_prototypes = summary['prototype'].tolist()
    df_major = rdf_df[rdf_df['prototype'].isin(major_prototypes)].copy()

    n_prototypes = len(major_prototypes)
    fig_height = max(3.0, 0.28 * n_prototypes + 1.2)
    fig, ax = plt.subplots(figsize=(4.5, fig_height))

    box_data = [df_major.loc[df_major['prototype'] == proto, metric].values for proto in major_prototypes]
    box = ax.boxplot(
        box_data,
        orientation='horizontal',
        patch_artist=True,
        widths=0.55,
        showfliers=False,
        medianprops={'color': '#222222', 'linewidth': 1.2},
        whiskerprops={'color': '#666666', 'linewidth': 0.8},
        capprops={'color': '#666666', 'linewidth': 0.8},
        boxprops={'linewidth': 0.8, 'edgecolor': '#444444'},
    )
    cmap = LinearSegmentedColormap.from_list('proto', ['#DDEBF7', '#3182BD'])
    for index, patch in enumerate(box['boxes']):
        patch.set_facecolor(cmap(index / max(len(box['boxes']) - 1, 1)))
        patch.set_alpha(0.95)

    ax.set_yticks(np.arange(1, n_prototypes + 1))
    ax.set_yticklabels([format_prototype_label(proto) for proto in major_prototypes])
    ax.set_xlabel(f'{metric_label} {metric_unit}')
    ax.set_title(f'Structure prototypes with n ≥ {min_prototype_count}', loc='left', pad=6)
    fig.tight_layout()

    png_path = output_dir / 'rdf_wasserstein_normalized_panel_c.png'
    fig.savefig(png_path, dpi=600, bbox_inches='tight', facecolor='white')
    plt.show()
    return png_path, summary


rdf_df = pd.read_csv(RDF_WASSERSTEIN_CSV)
panel_c_path, panel_c_summary = plot_panel_c_figure(
    rdf_df,
    OUTPUT_DIR,
    min_prototype_count=10,
    dataset_csv=DATA_CSV,
)

print('Saved', panel_c_path)
print(f'RDF rows: {len(rdf_df)}')
print('Prototype medians (n ≥ 10), high → low:')
print(
    panel_c_summary.sort_values('median', ascending=False)[
        ['prototype', 'n', 'median']
    ].to_string(index=False)
)